<a href="https://colab.research.google.com/github/prometricas/Yonny_Markov/blob/main/Yonny_finance.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# -*- coding: utf-8 -*-
# ============================================================
# VALORACIÓN DEL CAMBIO DE GENERACIÓN ENERGÉTICA (Carbón -> Gas)
# Enfoque: GBM con cambio de regímenes (Markov-switching) + Monte Carlo
# Fuente de datos:
#   - data_gas_carbón.xlsx  (serie histórica consolidada: gas y carbón)
#   - Flujo de caja proyecto 10082025.xlsx (flujos por escenario)
# ============================================================

# Yo dejo este notebook como una “pieza de exposición”: muchas gráficas, mucha trazabilidad
# y bloques reutilizables para iterar el modelo sin depender de @RISK ni EViews.

# -----------------------------
# (0) Instalación / imports
# -----------------------------
!pip -q install openpyxl statsmodels

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import statsmodels.api as sm
from statsmodels.tsa.regime_switching.markov_regression import MarkovRegression

import warnings
warnings.filterwarnings("ignore")

np.set_printoptions(suppress=True)


# -----------------------------
# (1) Carga de archivos
# -----------------------------
# Yo intento leer primero desde el directorio actual.
# Si no encuentro archivos, yo habilito carga manual en Colab.

if (not os.path.exists(SERIES_FILE)) or (not os.path.exists(FLOWS_FILE)):
    from google.colab import files
    uploaded = files.upload()

# Yo confirmo que los archivos quedaron disponibles.
print("¿Series existe?:", os.path.exists(SERIES_FILE), " -> ", SERIES_FILE)
print("¿Flujos existe?:", os.path.exists(FLOWS_FILE), " -> ", FLOWS_FILE)


# -----------------------------
# (2) Lectura de series (gas y carbón) desde una sola hoja
# -----------------------------
# Yo tomo el archivo consolidado: columnas esperadas = date, gas_price, carbon_price
df_prices = pd.read_excel(SERIES_FILE, sheet_name=0)

# Yo estandarizo fecha y orden.
df_prices["date"] = pd.to_datetime(df_prices["date"])
df_prices = df_prices.sort_values("date").reset_index(drop=True)

# Yo convierto la indexación a mensual “compatible” con modelos de series.
# (yo uso PeriodIndex mensual y lo paso a timestamp de fin de mes)
df_prices["month"] = df_prices["date"].dt.to_period("M").dt.to_timestamp("M")
df_prices = df_prices.drop(columns=["date"]).set_index("month")
df_prices = df_prices.asfreq("M")

# Yo elimino vacíos si existieran.
df_prices = df_prices.dropna()

display(df_prices.head())
display(df_prices.tail())


# -----------------------------
# (3) Gráficas descriptivas: niveles de precios
# -----------------------------
plt.figure()
plt.plot(df_prices.index, df_prices["gas_price"])
plt.title("Gas (Henry Hub) - Serie histórica (USD/MMBtu)")
plt.xlabel("Fecha")
plt.ylabel("Precio")
plt.grid(True)
plt.show()

plt.figure()
plt.plot(df_prices.index, df_prices["carbon_price"])
plt.title("Carbón (World Bank Pink Sheet) - Serie histórica (USD/ton)")
plt.xlabel("Fecha")
plt.ylabel("Precio")
plt.grid(True)
plt.show()

